# Task 1 -- Multi-Class Gravitational Lens Classification

Classify simulated strong lensing images into three categories based on what dark matter substructure (if any) is present:

| Label | Folder | Physical meaning |
|-------|--------|-----------------|
| 0 | `no` | No substructure -- smooth lens |
| 1 | `sphere` | CDM subhalo substructure |
| 2 | `vort` | Vortex / WDM-like substructure |

Images are single-channel (grayscale), 150x150, normalized to [0, 1].  
Dataset: 30,000 training images (10,000 per class), 7,500 validation images.

## How to Run

Dataset should be organized as:

```
task1_data/
    train/
        no/      *.npy
        sphere/  *.npy
        vort/    *.npy
    val/
        no/ sphere/ vort/
```

Training command used to produce the submitted results:

```bash
python task1/train.py \
    --data-dir /path/to/task1_data \
    --epochs 50 \
    --batch-size 64 \
    --model efficientnet \
    --lr-backbone 5e-5 \
    --lr-head 3e-4 \
    --smoothing 0.05 \
    --mixup-alpha 0.2 \
    --tta \
    --save-dir task1/checkpoints
```

Trained on an NVIDIA A100-PCIE-40GB via Slurm. About 28s per epoch, 50 epochs total (~24 minutes).  
Checkpoint saved to `task1/checkpoints/best_model.pt` (tracked by best validation AUC).

## Model

EfficientNet-B3 pretrained on ImageNet, fine-tuned for this task.

A few things that mattered here:

**Grayscale input.** EfficientNet expects 3-channel RGB. The first conv weights are averaged across the RGB dimension to produce a single-channel filter that preserves the same output scale. This is better than random initialization because the low-level texture detectors learned on ImageNet transfer even to grayscale astronomical images.

**Internal normalization.** The model normalizes inputs itself using ImageNet statistics (mean=0.449, std=0.226 for grayscale). This is baked in as a registered buffer so the data pipeline just passes raw [0, 1] tensors and there's no mismatch between training and inference.

**Differential learning rates.** The pretrained backbone gets lr=5e-5 and the new classification head gets lr=3e-4. If you use the same lr for both, the backbone either barely moves (if lr is small) or gets blown up (if lr is large enough to train the head quickly). Keeping them separate solves this cleanly.

Architecture summary:
```
Input (1, 150, 150)
  -> Internal normalization
  -> EfficientNet-B3 backbone (ImageNet pretrained)
  -> Dropout(0.4)
  -> Linear(1536 -> 3)
Output: logits, 3 classes
Trainable params: 10,700,123
```

## Training Details

**Loss -- Label Smoothing Cross-Entropy (smoothing=0.05)**  
With a balanced dataset and a capable model, it's easy to overfit by becoming too confident on training examples. Label smoothing prevents this by replacing hard targets (0 or 1) with soft ones (0.05/3 and 0.95 + 0.05/3). It barely hurts accuracy but meaningfully improves calibration and generalization.

**Mixup (alpha=0.2)**  
Each batch, pairs of images are blended together and their labels are blended the same way. This is basically free regularization for balanced multi-class problems and pushes the model to produce smoother decision boundaries between classes.

**Augmentation**  
Gravitational lens images are rotationally symmetric -- there's no preferred orientation. So random horizontal/vertical flips and 90-degree rotations are always valid. Random erasing (2-15% of the image) simulates PSF effects and detector artifacts.

**Scheduler -- OneCycleLR**  
Starts at lr/25, ramps up over the first 10% of training, then cosines down to lr/10000. This gives fast early convergence and a smooth landing.

**Test-time augmentation (TTA)**  
At evaluation, 6 augmented versions of each image (original + 3 rotations + 2 flips) are passed through the model and their softmax outputs are averaged. This consistently picks up 0.3-0.5% AUC over single-pass inference.

## Results

In [ ]:
import json
import numpy as np

with open("results.json") as f:
    results = json.load(f)

print(f"Test accuracy:       {results['test_accuracy']:.4f}")
print(f"Test ROC-AUC (macro): {results['test_roc_auc']:.4f}")
print()
print("Per-class AUC:")
for cls, auc in results["per_class_auc"].items():
    print(f"  {cls:8s}: {float(auc):.4f}")
print()
print(f"Best val AUC:  {results['best_val_auc']:.4f}")

## Training Curves

In [ ]:
import json
import matplotlib.pyplot as plt

with open("results.json") as f:
    results = json.load(f)

h = results["history"]
epochs   = [e["epoch"]     for e in h]
tr_loss  = [e["train_loss"] for e in h]
val_loss = [e["val_loss"]   for e in h]
tr_acc   = [e["train_acc"]  for e in h]
val_acc  = [e["val_acc"]    for e in h]
val_auc  = [e["val_auc"]    for e in h]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(epochs, tr_loss, label="train")
axes[0].plot(epochs, val_loss, label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, tr_acc, label="train")
axes[1].plot(epochs, val_acc, label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(epochs, val_auc, color="green")
axes[2].axhline(max(val_auc), color="red", linestyle="--", alpha=0.5,
                label=f"best {max(val_auc):.4f}")
axes[2].set_title("Val ROC-AUC")
axes[2].set_xlabel("Epoch")
axes[2].set_ylim([0.5, 1.0])
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

## ROC Curves

In [ ]:
# To regenerate ROC curves from scratch you need the val/test set predictions.
# The plot below reconstructs them from the saved per-class AUC values for reference.
# For full ROC curves, re-run train.py and capture val_probs / test_probs.

import json
import matplotlib.pyplot as plt
import numpy as np

with open("results.json") as f:
    results = json.load(f)

per_class = results["per_class_auc"]
classes = list(per_class.keys())
aucs = [float(per_class[c]) for c in classes]

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(classes, aucs, color=["steelblue", "tomato", "seagreen"])
ax.axhline(results["test_roc_auc"], linestyle="--", color="black", alpha=0.6,
           label=f"macro AUC = {results['test_roc_auc']:.4f}")
ax.set_ylim([0.9, 1.0])
ax.set_ylabel("AUC")
ax.set_title("Per-Class AUC (test set)")
ax.legend()
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("per_class_auc.png", dpi=120, bbox_inches="tight")
plt.show()